# 02 — FFT and envelope spectrum
A bearing defect produces a short impact every time a ball passes it. The impact excites high-frequency structural resonances, so the plain FFT shows energy in a broad high band, not at the fault frequency. Envelope analysis band-passes that resonance region, takes the Hilbert envelope (the impact repetition), and the spectrum of the envelope shows the fault frequency directly.

In [ ]:
import sys; sys.path.insert(0, '../src')
import numpy as np, matplotlib.pyplot as plt
import bearing as b
plt.rcParams['figure.dpi'] = 110

In [ ]:
ff = b.fault_freqs(1797)
{k: round(v, 1) for k, v in ff.items()}

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(9, 7), sharex=True)
for ax, num in zip(axes, [97, 105, 118, 130]):
    x, _ = b.load(num)
    f = np.fft.rfftfreq(len(x), 1 / b.FS)
    ax.semilogy(f, np.abs(np.fft.rfft(x * np.hanning(len(x)))), lw=0.5)
    ax.axvspan(2000, 5000, color='orange', alpha=0.15)
    ax.set_title(b.FILES[num][0], fontsize=9, loc='left')
axes[-1].set_xlabel('frequency [Hz]  (shaded: band used for the envelope)')
fig.tight_layout(); fig.savefig('../images/01-fft.png')

In [ ]:
colors = {'BPFO': 'tab:red', 'BPFI': 'tab:green', 'BSF': 'tab:purple'}
fig, axes = plt.subplots(4, 1, figsize=(9, 8), sharex=True)
for ax, num in zip(axes, [97, 105, 118, 130]):
    x, rpm = b.load(num)
    f, a = b.envelope_spectrum(x)
    m = f < 500
    ax.plot(f[m], a[m], lw=0.7, color='k')
    for k, c in colors.items():
        ax.axvline(b.fault_freqs(rpm)[k], color=c, ls='--', lw=1, label=k)
    ax.set_title(b.FILES[num][0], fontsize=9, loc='left')
axes[0].legend(fontsize=8, ncol=3)
axes[-1].set_xlabel('frequency [Hz]')
fig.tight_layout(); fig.savefig('../images/01-envelope-spectrum.png')

Inner race → peak at BPFI (with shaft-rate sidebands, since the inner race rotates through the load zone). Outer race → clean peak at BPFO and harmonics. Ball fault → weak, smeared peaks: the ball spins and only sometimes hits the defect against a race, which is why ball faults are the hardest class in CWRU.